# Synthetic rough-volatility hedging: L4 recipe study

This notebook is the controlled laboratory of the project. Every market path comes from a rough-Bergomi simulator with known parameters ($H = 0.1$, $\eta = 1$, $\rho = -0.5$, $\xi_0 = 0.1$, 50 rebalancing dates over 30 days). Because the true model is known, any difference between hedges is attributable to the hedging rule and to how it was trained, not to model error. Notebook 03 then asks the harder question of what survives on real market data.

**The hedging problem.** A trader sells one at-the-money European call ($K = S_0 = 1$) and trades the underlying at 50 dates, paying 10 bp of notional on every trade. The terminal loss of a strategy $\delta = (\delta_0, \dots, \delta_{49})$ is

$$L(\delta) = (S_T - K)^+ - \sum_{k} \delta_k (S_{k+1} - S_k) + c \sum_{k} |\delta_k - \delta_{k-1}| \, S_k ,$$

so a lower $L$ is better. The risk measure is the conditional value-at-risk at 95%, the mean of the worst 5% of losses:

$$\mathrm{CVaR}_{95}(L) = \min_{v} \Big\{ v + \tfrac{1}{0.05} \, \mathbb{E}\big[(L - v)^+\big] \Big\}.$$

Learned hedgers minimise exactly this Rockafellar–Uryasev objective, so the minimiser $v$ is also learned: it should converge to the 95% value-at-risk, and Figure 5.6 checks that it does.

**What is compared.**

| Factor | Levels |
|---|---|
| Training recipe | R-GPU (large batch), R-NTBN (one full-batch step per epoch, after a no-transaction-band study), R-SIG (AdamW with cosine schedule, after a signature study), R-DH (the original deep-hedging settings), R-HOR (the rough-volatility deep-hedging settings) |
| Architecture | FFN (feed-forward on the current state), GRU (recurrent, sees the path history), sig2 (depth-2 path signature features) |
| Seeds | 3 per cell |
| Training profile | small (paper path counts $\times 0.1$) and large (8 to 25 times more paths per step, sized for one NVIDIA L4) |
| Classical baselines | Black–Scholes delta, Leland (cost-adjusted volatility), Whalley–Wilmott no-trade band |

That gives 5 × 3 × 3 = 45 trained units per profile, 90 in total, all scored on the **same 20,000 held-out test paths**. Sharing the test paths makes every comparison paired: the difference between two hedges is measured path by path, which removes most of the Monte Carlo noise from the comparison.


In [1]:
from pathlib import Path
import json

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
study = json.loads((ROOT / 'results/nb02/t4_stats.json').read_text())
print('Bootstrap replicates:', study['n_boot'])
print('Comparison unit:', study['holm_family'])

Bootstrap replicates: 2000
Comparison unit: 15 recipe x arm cells per profile


## 1. Headline result

* **Learned hedgers beat Black–Scholes delta in every large-profile cell.** All 15 recipe × architecture cells have a lower CVaR95 than delta after a Holm correction over the 15 tests. The best unit, `R-HOR_GRU_2`, reaches CVaR95 = 0.05742 against 0.06195 for delta, a paired gap of −0.00453 (95% bootstrap interval −0.00503 to −0.00400). That is a 7.3% reduction of tail risk on identical paths.
* **More data per step helps.** The large profile is better than the small profile in all 15 cells: mean CVaR95 0.05855 against 0.05975, a 2.0% improvement (paired $t = 5.16$, $p = 0.00015$).
* **The training recipe matters far more than the architecture.** In the large profile the recipe explains 92.8% of the variance in CVaR95, the architecture under 1% (Figure 5.11).
* **The learned tail threshold is calibrated.** At the last epoch the learned $v$ sits within a median 0.5% of the empirical VaR95 (maximum 2.2%).

These statements are conditional on the simulator, the 10 bp cost and the compute given to each recipe. They say which training choices work when the model is right; they do not yet say anything about real markets.


## 2. CVaR95 leaderboard

![Figure 5.1: CVaR95 leaderboard](../figures/fig-5.1.png)

**What it shows.** Each row is one recipe × architecture cell; the dot is the mean test CVaR95 over three seeds and the bar is a 95% $t$-interval across those seeds. The dashed vertical lines are the three classical hedges: delta (0.0620), Leland (0.0614) and the Whalley–Wilmott band (0.0636). The left panel is the small profile, the right panel the large profile.

**How to read it.** Further left is better. A cell whose whole interval lies left of the delta line beats delta consistently across seeds; a wide bar means the result depends on the random seed.

**What we find.**
* In the large profile every cell lies left of all three classical lines. R-HOR and R-DH cluster at the left edge (about 0.0576 to 0.0580), R-SIG and R-GPU follow, and R-NTBN is clearly last (about 0.0602 to 0.0605), just inside the classical range.
* The small profile has the same ordering but much wider seed intervals; SIG/sig2 and DH/FFN even straddle the delta line. With 10× fewer paths, the outcome of a single training run is unreliable.
* Leland barely improves on delta, and the no-trade band is the worst tail hedge of the three: it saves trading cost but leaves larger hedging errors in bad paths (see Figure 5.14).

**Why it matters.** The ranking is driven by the recipe rows, not the colours: reading down the right panel, the colour groups (architectures) move together. That is the visual version of the variance split in Figure 5.11.

*Data:* `../figures/fig-5.1.csv`


## 3. Paired CVaR gaps to delta

![Figure 5.2: paired CVaR gaps](../figures/fig-5.2.png)

**What it shows.** A heatmap of $\Delta = \mathrm{CVaR}_{95}(\text{learned}) - \mathrm{CVaR}_{95}(\text{delta})$ for every cell. Each seed is compared with delta by a paired bootstrap on the 20,000 shared test paths (2,000 resamples); the cell value is the mean of the three per-seed estimates. The cell text gives the estimate and the Holm-adjusted $p$-value over the 15 cells of the profile.

**How to read it.** Blue means the learned hedge has the lighter tail. The Holm adjustment controls the probability of any false claim across the whole grid, so $p = 0.015$ here is a family-wise statement, not a per-cell one. It is also the smallest value the procedure can reach with 2,000 resamples and 15 tests, which is why it repeats.

**What we find.**
* **Large profile:** all 15 cells are significantly below delta. The gap grows from R-NTBN (about −0.0015 to −0.0018) to R-HOR (−0.0041 to −0.0044).
* **Small profile:** 12 of 15 cells remain significant. The three exceptions are SIG/FFN ($p = 0.354$), SIG/sig2 ($p = 0.367$) and NTBN/sig2 ($p = 0.171$), exactly the cells with the widest seed bars in Figure 5.1.

**Why it matters.** This is the primary inferential panel. The leaderboard shows levels; this figure shows that the improvement is a property of the strategies on the same scenarios, not luck in which paths were drawn.

*Data:* `../figures/fig-5.2.csv`


## 4. Validation learning curves

![Figure 5.3: validation learning curves](../figures/fig-5.3.png)

**What it shows.** Validation CVaR95 against optimizer steps for each recipe (large profile), as the median over three seeds with a min–max band, one colour per architecture.

**How to read it.** A curve that is still falling at the right edge was stopped too early; a flat tail means the budget was enough. The horizontal extent differs because the recipes take different numbers of steps: about 500 for R-GPU, 120 for R-NTBN, 1,600 for R-SIG, up to 2,000 for R-DH and 2,325 for R-HOR.

**What we find.**
* Every recipe except R-NTBN drops below 0.065 within roughly the first 10% of its steps, then improves slowly. The last few basis points of notional come from the long tail of training, which is why the longest recipes (R-DH, R-HOR) finish best.
* R-NTBN is different: with one full-batch step per epoch it needs about 70 of its 120 steps just to settle, and its seeds disagree strongly early on (wide bands). It ends where the others are after a few hundred steps, so it is under-trained, not a worse idea.
* R-DH stops early on some seeds (its curves end before 2,000 steps) because of its patience rule.

**Why it matters.** Differences between recipes are largely differences in effective optimisation budget. Figure 5.4 makes the cost of that budget explicit.

*Data:* `../figures/fig-5.3.csv`


## 5. Compute–risk frontier

![Figure 5.4: compute risk frontier](../figures/fig-5.4.png)

**What it shows.** Final test CVaR95 of every unit against the GPU-seconds it consumed (sum of epoch wall times, log scale). Marker shape is the recipe, colour the architecture, filled markers the large profile and hollow markers the small profile.

**How to read it.** The lower-left corner is the efficient frontier: low risk for little compute. Points to the right paid more GPU time; points higher up got less for it.

**What we find.**
* The cloud of points slopes down and to the right: more compute buys lower tail risk, but with diminishing returns on a log scale. Median GPU-seconds per large unit are about 84 for R-NTBN, 206 for R-GPU, 420 for R-SIG, 437 for R-DH and 727 for R-HOR.
* R-GPU sits close to the frontier: at about 28% of R-HOR's compute its cells end within about 0.0006 to 0.0013 of the best CVaR95.
* Small-profile points (hollow) are cheaper but both higher and more scattered, which is the seed instability seen in Figure 5.1.

**Why it matters.** If compute is constrained, a large-batch recipe like R-GPU is the sensible default; the extra 6 to 13 basis points of notional in CVaR95 from R-HOR cost about 3.5 times the GPU time.

*Data:* `../figures/fig-5.4.csv`


## 6. Gradient noise scale against batch size

![Figure 5.5: gradient noise](../figures/fig-5.5.png)

**What it shows.** The simple gradient noise scale per epoch (log scale), with the batch size of each recipe as a dashed line:

$$B_{\text{simple}} = \frac{\operatorname{tr}\,\Sigma}{\lVert G \rVert^2} \approx B \cdot \frac{\mathbb{E}\lVert g_b \rVert^2 - \lVert \bar g \rVert^2}{\lVert \bar g \rVert^2},$$

where $g_b$ are the per-step mini-batch gradients within an epoch, $\bar g$ is their mean and $B$ the batch size. $B_{\text{simple}}$ estimates the critical batch size: below it, larger batches make each step proportionally more useful; above it, extra samples are mostly wasted. R-NTBN takes one step per epoch, so the within-epoch spread, and hence $B_{\text{simple}}$, is undefined.

**What we find.** After the first few epochs the noise scale is 100 to 900 times the batch size in every recipe (medians: R-GPU about 106×, R-SIG 214×, R-DH 374×, R-HOR 875×). The tail-risk objective is extremely noisy: only 5% of each batch contributes to the CVaR gradient, so the useful signal per sample is small.

**Why it matters.** Every recipe is far below its critical batch, so even the large profile is batch-starved. This explains why larger batches helped in all 15 cells (Figure 5.10), and predicts that still larger batches, which an L4 can hold, would keep paying off. It is the most actionable training finding of the study.

*Data:* `../figures/fig-5.5.csv`


## 7. Calibration of the learned tail threshold

![Figure 5.6: CVaR threshold](../figures/fig-5.6.png)

**What it shows.** Two curves per architecture and recipe: the learned Rockafellar–Uryasev threshold $v$ (solid) and the empirical VaR95 of the current policy on a fixed 4,096-path probe set (dashed), median over seeds.

**How to read it.** At the optimum of the CVaR objective, $v$ equals the 95% quantile of the loss. Solid and dashed lines meeting means the network has learned not only a hedge but also a correct estimate of its own tail.

**What we find.**
* For R-GPU, R-SIG, R-DH and R-HOR the two lines lock together within about five epochs and stay together. The final gap is a median 0.5% across the 45 large units, at most 2.2%.
* R-NTBN shows the slow, oscillating approach expected from one step per epoch: $v$ overshoots to about 0.12 to 0.13 around epoch 25 and only converges after about epoch 70.

**Why it matters.** A calibrated $v$ is a free by-product of CVaR training: the model reports its own 95% loss quantile. It also confirms that the optimisation reached the intended objective rather than a degenerate solution.

*Data:* `../figures/fig-5.6.csv`


## 8. How far the learned policy moves from delta

![Figure 5.7: policy distance](../figures/fig-5.7.png)

**What it shows.** The mean absolute difference between the learned hedge ratio $\delta_\theta$ and the Black–Scholes delta $\delta_{BS}$ (using the model's average volatility $\sqrt{\xi_0}$), over all dates (left) and at the first date only (right), as a function of epoch. Line style is the recipe, colour the architecture.

**How to read it.** Zero means the network has rediscovered delta. A stable positive distance means a systematically different hedge.

**What we find.**
* All recipes converge to a distance of about 0.056 to 0.068 over all dates (medians at the last epoch), not to zero. The learned policies stay close to delta, but differ by about 6 percentage points of the hedge ratio on average.
* At the first date the distance is smaller, about 0.03 to 0.04: at inception the learned hedge is almost delta. The deviations build up later, when the path history and the rough volatility state carry information that Black–Scholes ignores.
* R-NTBN wanders far from delta for its first 60 to 80 epochs before settling, consistent with Figures 5.3 and 5.6.

**Why it matters.** The tail improvement does not come from an exotic strategy. It comes from small, state-dependent corrections around delta; that is also why these hedges are plausible to deploy and to explain.

*Data:* `../figures/fig-5.7.csv`


## 9. Loss-tail evolution during training

![Figure 5.8: loss survival](../figures/fig-5.8.png)

**What it shows.** For the best recipe (R-HOR, seed 0), the survival function $P(L > x)$ of the probe-set losses at five snapshots during training, on a log vertical axis, one panel per architecture.

**How to read it.** A curve further left, or one that falls faster, has a thinner tail. On a log scale the far right of each curve is the 1-in-1,000 to 1-in-4,000 region.

**What we find.**
* The untrained network (epoch 0, dark) has a heavy tail: about 10% of probe paths lose more than 0.06.
* By epoch 18 the tail has collapsed onto its final shape: fewer than 1% of paths lose more than 0.065. Later snapshots (epochs 37, 56, 74) barely move, so most of the tail risk reduction happens in the first quarter of training.
* The extreme right end (the last few paths) still moves between snapshots, especially for the GRU. The worst handful of paths is where the remaining training effort goes, and where results are least stable.

**Why it matters.** It shows how CVaR training works: it reshapes the tail first and leaves the body of the distribution largely in place (see Figure 5.9).

*Data:* `../figures/fig-5.8.csv`


## 10. P&L quantile fan

![Figure 5.9: P and L quantiles](../figures/fig-5.9.png)

**What it shows.** Quantiles of the probe-set P&L (the negative of the loss) over training for R-HOR, seed 0: median (black), and the 25–75%, 5–95% and 1–99% bands.

**How to read it.** Since P&L $= -L$, the lower edge of the fan is the loss tail. A fan that narrows from below means the bad outcomes are being removed; a median that moves means the typical outcome changes too.

**What we find.**
* The lower 1% band rises from about −0.07 to −0.08 at epoch 0 to about −0.06 within ten epochs and then holds steady.
* The median stays near −0.038 throughout. That is the premium-free cost of hedging the short call, which no strategy can remove.
* The three architectures produce nearly identical fans: once trained by the same recipe, they converge to very similar distributions.

**Why it matters.** The CVaR objective buys a better worst case without degrading the typical outcome. A hedger that only trimmed the tail by accepting a worse median would show the black line falling; it does not.

*Data:* `../figures/fig-5.9.csv`


## 11. Small against large training profile

![Figure 5.10: profile comparison](../figures/fig-5.10.png)

**What it shows.** For each of the 15 cells, the mean test CVaR95 of the small profile (hollow, left) joined to the large profile (filled, right), with 95% seed intervals.

**How to read it.** A line sloping down is an improvement from more data per step; shorter error bars on the right mean the result became more reproducible.

**What we find.**
* All 15 lines slope down: on average CVaR95 falls by 0.00121 (2.0%), with paired $t = 5.16$, $p = 0.00015$.
* The largest gains are in R-SIG, which went from 1,000 to 25,000 training paths: its CVaR95 falls by about 0.002 to 0.003, and its seed intervals shrink from the widest of the study to among the narrowest.
* R-NTBN gains least, because its constraint is the number of steps (120), not the data per step.

**Why it matters.** Together with Figure 5.5, this is the case for using a GPU like the L4 as intended: larger batches are not just faster, they produce better and more reproducible tail hedges for this objective.

*Data:* `../figures/fig-5.10.csv`


## 12. What drives tail risk: variance attribution

![Figure 5.11: variance attribution](../figures/fig-5.11.png)

**What it shows.** A two-way analysis of variance of CVaR95 across the 45 units of each profile, split into recipe, architecture, their interaction and the seed residual (type II sums of squares, as shares of the total).

**How to read it.** Each bar sums to 100%. The larger a block, the more of the spread in outcomes that factor explains.

**What we find.**

| Profile | Recipe | Architecture | Interaction | Seed (residual) |
|---|---|---|---|---|
| small | 71.0% | about 2% | 4.7% | 22.0% |
| large | 92.8% | under 1% | about 3% | 4.0% |

**Why it matters.** Two conclusions follow. First, the choice of training recipe matters far more than the choice among FFN, GRU and signature architectures: architecture debates are second-order here. Second, more data per step cuts the share of outcome variance due to random seeds from 22% to 4%. Single-seed results are therefore far more trustworthy in the large profile, which matters because most published deep-hedging results report one seed.

*Data:* `../figures/fig-5.11.csv`


## 13. L4 utilisation timeline

![Figure 5.12: L4 utilization timeline](../figures/fig-5.12.png)

**What it shows.** The `nvidia-smi` record of the final large-profile pass, sampled every 2 seconds: GPU utilisation with the number of training units in flight shaded (top), memory in use (middle) and power draw (bottom).

**How to read it.** A well-used accelerator shows utilisation near 100% while units are running, memory close to its capacity and power near the 72 W limit of the L4.

**What we find.**
* Up to 16 units ran concurrently through a scheduler that packed processes by their measured memory peak.
* Utilisation stayed at 98–100% for about nine minutes. Memory peaked at about 18.5 GiB of the 22 GiB available, and power ran at 58–67 W.
* As units finished, memory and power stepped down, and utilisation fell once only a few long units remained.

**Why it matters.** This is the evidence that the study used the hardware it claims to have used. The alignment between the unit count (from the scheduler log) and the GPU clock is approximate, so the shading is indicative rather than exact.

*Data:* `../figures/fig-5.12.csv`


## 14. Which layers learn

![Figure 5.13: gradient share by layer](../figures/fig-5.13.png)

**What it shows.** For R-HOR, seed 0, the share of the total gradient norm carried by each parameter group over the epochs: the three linear layers of the FFN, the recurrent cell and output head of the GRU, and the two head layers of the signature model.

**How to read it.** Brighter cells receive a larger share of each update; a stable pattern means a stable division of labour inside the network.

**What we find.**
* **GRU:** the recurrent cell carries about 75–85% of the gradient throughout. Most of the learning happens in how the path history is summarised, not in the final mapping to a hedge ratio.
* **sig2:** the signature transform has no trainable weights, so all learning is in the head, with about two thirds in the last layer.
* **FFN:** the gradient spreads over its layers, with the first layer receiving the smallest share.
* All patterns are established within the first epoch or two and barely change afterwards.

**Why it matters.** It explains why the GRU can use path information that the FFN cannot see, and shows that none of the three architectures has a dead or saturated layer during training.

*Data:* `../figures/fig-5.13.csv`


## 15. Tail risk against trading activity

![Figure 5.14: risk against turnover](../figures/fig-5.14.png)

**What it shows.** Test CVaR95 against mean turnover (total absolute shares traded per path, including entry and exit) for every unit, plus the three classical hedges as stars.

**How to read it.** Lower is less tail risk; further left is less trading and therefore lower cost. A hedge that is both lower and further left than another dominates it.

**What we find.**
* Delta (turnover 3.13) and Leland (3.10) trade the most. The Whalley–Wilmott band trades least (1.52) but has the worst tail (0.0636).
* The learned hedges sit in between, at a turnover of about 2.2 to 2.7. The best large-profile units combine about 25% less trading than delta (for example 2.34 for `R-HOR_GRU_2`) with a lower CVaR95.
* They therefore dominate delta and Leland on both axes, and trade off against the band: more trading than the band, much less tail risk.

**Why it matters.** The learned policies have discovered a cost-aware compromise: they trade less than continuous delta hedging but do not leave the large exposure a fixed band allows in fast markets.

*Data:* `../figures/fig-5.14.csv`


## 16. Full loss distribution and the extreme tail

![Figure 5.15: loss distribution](../figures/fig-5.15.png)

**What it shows.** Exceedance curves $P(L > x)$ on all 20,000 test paths for the best learned unit (`R-HOR_GRU_2`) and the three classical hedges, with a zoom on the region beyond delta's 90% loss quantile.

**How to read it.** Below another curve means fewer paths exceed that loss. The far right shows individual worst-case paths, so it is noisy.

**What we find.**
* From the 90% to roughly the 99.9% level the learned hedge is clearly better. Its CVaR at 99% is 0.0696 against 0.0760 for delta, and its 99.9% quantile is 0.0833 against 0.0870.
* The single worst path tells the opposite story: the learned hedge loses 0.194 on it against 0.127 for delta. The training objective averages over the worst 5% and gives little weight to one path in 20,000.

**Why it matters.** It is an honest limitation of CVaR training. The learned hedge improves the tail that the objective targets, but it does not bound the worst case. A risk manager who also cares about the maximum loss would add a constraint or a higher-level tail term.

*Data:* `../figures/fig-5.15.csv`


## 17. Optimisation health: gradient norm and update size

![Figure 5.16: gradient and update diagnostics](../figures/fig-5.16.png)

**What it shows.** The mean gradient norm (left) and the update-to-weight ratio $\lVert \Delta\theta \rVert / \lVert \theta \rVert$ per epoch (right), both on log scales, median over seeds, for every recipe and architecture.

**How to read it.** Healthy training has gradients that neither explode nor vanish, and update ratios that decay smoothly from about $10^{-1}$ to $10^{-2}$ or $10^{-3}$ as training settles. A ratio that is flat and large signals a learning rate that is too high; one that collapses early signals stalled learning.

**What we find.**
* Gradient norms stay between about $10^{-2}$ and $10^{-1}$ for most of training, with no explosions.
* R-NTBN shows sharp dips to $10^{-3}$ in the GRU and sig2 curves around epochs 15–40. These coincide with the plateau of its learning curve in Figure 5.3.
* Update ratios fall smoothly. The cosine schedule of R-SIG and the decaying gradients of the long recipes bring them to about $3 \times 10^{-3}$ by the end, while R-DH, with a constant learning rate, stays nearer $10^{-2}$.

**Why it matters.** No recipe failed for numerical reasons. The differences in Figure 5.1 reflect optimisation budget and noise, not broken training.

*Data:* `../figures/fig-5.16.csv`


In [2]:
required = [ROOT / 'figures' / f'fig-5.{n}.{ext}' for n in range(1, 17) for ext in ('png', 'csv')]
missing = [p.name for p in required if not p.exists()]
assert not missing, f'Missing compact study artifacts: {missing}'
print(f'Validated {len(required)} committed figure artifacts.')

Validated 32 committed figure artifacts.


## 18. Limits and reproduction

**Limits.**
* All evidence here is conditional on one simulated market ($H = 0.1$, $\eta = 1$, $\rho = -0.5$), one contract (an at-the-money 30-day call), one cost level (10 bp) and the compute given to each recipe. Notebook 03 shows how much of it survives contact with real data.
* Three seeds per cell are enough for the paired path-level tests, which use 20,000 paths, but they make the across-seed intervals of Figure 5.1 coarse.
* The recipes adapt published training settings to a common market and common architectures. They are faithful to the papers in optimizer, learning rate, schedule and epochs, but not in network width or time grid.
* No model weights are kept; the statistics and the exact source code that produced them are.

**Reproduction.** `scripts/t4_recipes.py` (small profile) and `scripts/t4_recipes_large.py` (large profile) train the units; `scripts/gen_figures_t4.py` regenerates Figures 5.1–5.16 from the per-unit result files; `results/nb02/t4_stats.json` holds every number quoted above.
